# 03 KPIs Gold

Builds the Gold KPI views from Silver:

| View | Grain | What it answers |
|---|---|---|
| `gold_payments_monthly` | month | Payment attempts, settled, failed, gross revenue (settled amount) |
| `gold_chargeback_rate` | month | Chargebacks opened in the month divided by settled payments in the month |
| `gold_revenue_by_plan` | month, plan | Gross revenue, revenue later lost to chargebacks, net revenue |
| `gold_kpi_monthly` | month | One row per month joining the two monthly views, plus failure rate |

Views, not tables: they always reflect the current Silver data and cost nothing to store. The last cell prints each view and returns them as JSON (the runner script reads that).

In [1]:
dbutils.widgets.text("CATALOG", "")
dbutils.widgets.text("SCHEMA", "demo_finance")

CATALOG = dbutils.widgets.get("CATALOG").strip() or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("SCHEMA").strip()
if not SCHEMA:
    raise ValueError("SCHEMA must not be empty")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")
print(f"schema: {CATALOG}.{SCHEMA}")

schema: <catalog>.demo_finance_runnable


In [2]:
# 1) Monthly payments summary (counts + gross revenue)
spark.sql("""
CREATE OR REPLACE VIEW gold_payments_monthly
COMMENT 'Gold KPI: monthly payment counts and gross revenue (settled amount) from silver_payments.'
AS
SELECT
  month,
  COUNT(*)                                                   AS payments,
  SUM(CASE WHEN status = 'settled' THEN 1 ELSE 0 END)        AS settled,
  SUM(CASE WHEN status = 'failed'  THEN 1 ELSE 0 END)        AS failed,
  ROUND(SUM(CASE WHEN status = 'settled' THEN amount ELSE 0 END), 2) AS gross_revenue
FROM silver_payments
GROUP BY month
""")
print("created view gold_payments_monthly")

created view gold_payments_monthly


In [3]:
# 2) Monthly chargeback rate (chargebacks opened in month / payments settled in month)
spark.sql("""
CREATE OR REPLACE VIEW gold_chargeback_rate
COMMENT 'Gold KPI: monthly chargebacks and chargeback rate (chargebacks opened / settled payments, same month).'
AS
WITH cbs AS (
  SELECT month_opened AS month, COUNT(*) AS chargebacks
  FROM silver_chargebacks
  GROUP BY month_opened
),
pays AS (
  SELECT month, SUM(CASE WHEN status = 'settled' THEN 1 ELSE 0 END) AS settled
  FROM silver_payments
  GROUP BY month
)
SELECT
  COALESCE(p.month, c.month) AS month,
  COALESCE(c.chargebacks, 0) AS chargebacks,
  COALESCE(p.settled, 0)     AS settled,
  CASE WHEN COALESCE(p.settled, 0) = 0 THEN 0
       ELSE ROUND(COALESCE(c.chargebacks, 0) / p.settled, 4)
  END AS chargeback_rate
FROM pays p
FULL OUTER JOIN cbs c
  ON p.month = c.month
""")
print("created view gold_chargeback_rate")

created view gold_chargeback_rate


In [4]:
# 3) Revenue by plan: gross (settled), lost to chargebacks, net.
#    Losses are attributed to the month of the original payment.
spark.sql("""
CREATE OR REPLACE VIEW gold_revenue_by_plan
COMMENT 'Gold KPI: monthly settled revenue by plan, revenue later lost to chargebacks, and net revenue.'
AS
WITH settled AS (
  SELECT p.month, s.plan, p.payment_id, p.amount
  FROM silver_payments p
  JOIN silver_subscriptions s
    ON p.customer_id = s.customer_id
  WHERE p.status = 'settled'
),
lost AS (
  SELECT DISTINCT payment_id
  FROM silver_chargebacks
  WHERE outcome = 'lost'
)
SELECT
  st.month,
  st.plan,
  COUNT(*)                                                         AS settled_payments,
  ROUND(SUM(st.amount), 2)                                         AS gross_revenue,
  ROUND(SUM(CASE WHEN l.payment_id IS NOT NULL THEN st.amount ELSE 0 END), 2) AS chargeback_losses,
  ROUND(SUM(CASE WHEN l.payment_id IS NULL THEN st.amount ELSE 0 END), 2)     AS net_revenue
FROM settled st
LEFT JOIN lost l
  ON st.payment_id = l.payment_id
GROUP BY st.month, st.plan
""")
print("created view gold_revenue_by_plan")

created view gold_revenue_by_plan


In [5]:
# 4) Convenience: one row per month with everything
spark.sql("""
CREATE OR REPLACE VIEW gold_kpi_monthly
COMMENT 'Gold KPI: unified monthly KPIs (payments, settled, failed, gross_revenue, chargebacks, chargeback_rate, failure_rate).'
AS
SELECT
  COALESCE(p.month, r.month) AS month,
  p.payments,
  p.settled,
  p.failed,
  p.gross_revenue,
  r.chargebacks,
  r.chargeback_rate,
  CASE WHEN COALESCE(p.payments, 0) = 0 THEN 0
       ELSE ROUND(COALESCE(p.failed, 0) / p.payments, 4)
  END AS failure_rate
FROM gold_payments_monthly p
FULL OUTER JOIN gold_chargeback_rate r
  ON p.month = r.month
""")
print("created view gold_kpi_monthly")

created view gold_kpi_monthly


In [6]:
import json

GOLD = {
    "gold_payments_monthly": "SELECT * FROM gold_payments_monthly ORDER BY month",
    "gold_chargeback_rate":  "SELECT * FROM gold_chargeback_rate ORDER BY month",
    "gold_revenue_by_plan":  "SELECT * FROM gold_revenue_by_plan ORDER BY month, plan",
    "gold_kpi_monthly":      "SELECT * FROM gold_kpi_monthly ORDER BY month",
}
results = {}
for name, sql in GOLD.items():
    df = spark.sql(sql)
    rows = [r.asDict() for r in df.collect()]
    results[name] = {"columns": df.columns, "rows": rows}
    print(f"{name}: {len(rows)} rows")
    df.show(len(rows), truncate=False)

totals = spark.sql("""
  SELECT ROUND(SUM(gross_revenue), 2) AS gross_revenue,
         ROUND(SUM(chargeback_losses), 2) AS chargeback_losses,
         ROUND(SUM(net_revenue), 2) AS net_revenue
  FROM gold_revenue_by_plan
""").first().asDict()
print("Revenue totals (all months, all plans):", totals)
results["totals"] = totals

gold_payments_monthly: 18 rows
+-------+--------+-------+------+-------------+
|month  |payments|settled|failed|gross_revenue|
+-------+--------+-------+------+-------------+
|2024-01|309     |285    |24    |6257.15      |
|2024-02|408     |378    |30    |8476.22      |
|2024-03|502     |465    |37    |10225.35     |
|2024-04|553     |520    |33    |11424.8      |
|2024-05|624     |569    |55    |12404.31     |
|2024-06|655     |606    |49    |13253.94     |
|2024-07|703     |656    |47    |14333.44     |
|2024-08|727     |679    |48    |14863.21     |
|2024-09|747     |699    |48    |15343.01     |
|2024-10|789     |713    |76    |15572.87     |
|2024-11|803     |741    |62    |16302.59     |
|2024-12|807     |772    |35    |16962.28     |
|2025-01|839     |787    |52    |17552.13     |
|2025-02|845     |786    |59    |17562.14     |
|2025-03|871     |795    |76    |17602.05     |
|2025-04|842     |765    |77    |17032.35     |
|2025-05|814     |755    |59    |16852.45     |
|2025-06|

In [7]:
# Return the Gold results to the caller (the runner prints them).
dbutils.notebook.exit(json.dumps(results, default=str))